# Prompt Flows in RAG

This notebook covers a few simple prompt-flow patterns used in retrieval-augmented generation:

- Query rewriting prompt flow
- Contextual compression
- Structured prompt templates for retrieval augmentation

LangChain’s agentic RAG guide includes a dedicated question-rewrite step, and the retrieval docs describe hybrid RAG as a pipeline with query enhancement, retrieval validation, and answer validation. The Deep Agents context-engineering docs describe context compression as a way to manage growing context, and LangSmith prompt docs explain the difference between prompts and reusable prompt templates.


## Learning goals

By the end of this notebook, you should be able to:

1. Rewrite a user question before retrieval.
2. Compress retrieved context into a smaller, cleaner prompt.
3. Build reusable prompt templates for retrieval augmentation.
4. See how these ideas fit into hybrid RAG.


## 1) Install packages

```bash
pip install -U langchain langchain-community langchain-text-splitters langchain-huggingface faiss-cpu sentence-transformers
```

If you later want to run this with a model, you can add a provider package such as `langchain-groq`, `langchain-openai`, or `langchain-ollama`.


In [ ]:
%pip install -qU langchain langchain-community langchain-text-splitters langchain-huggingface faiss-cpu sentence-transformers


## 2) Build a tiny toy knowledge base

We will use a few short passages so the prompt flows stay easy to inspect.


In [ ]:
from langchain_core.documents import Document

docs = [
    Document(
        page_content='LangChain supports retrieval pipelines built from loaders, splitters, embeddings, vector stores, and retrievers.',
        metadata={'source': 'kb_1', 'topic': 'retrieval'},
    ),
    Document(
        page_content='Hybrid RAG can include query enhancement, retrieval validation, and answer validation before returning a final answer.',
        metadata={'source': 'kb_2', 'topic': 'rag'},
    ),
    Document(
        page_content='Context compression helps reduce the amount of retrieved text passed into the model while keeping the important parts.',
        metadata={'source': 'kb_3', 'topic': 'compression'},
    ),
    Document(
        page_content='Prompt templates make prompts reusable by replacing placeholders with runtime variables.',
        metadata={'source': 'kb_4', 'topic': 'prompting'},
    ),
]

docs


## 3) Query rewriting

A rewrite step improves retrieval when the original user question is vague, short, or underspecified.

A simple rewrite flow usually:
- keeps the user’s intent
- adds missing keywords
- makes the retrieval query more explicit

LangChain’s agentic RAG tutorial includes a dedicated “rewrite question” step, and the retrieval docs describe query enhancement as a key part of hybrid RAG.


In [ ]:
def rewrite_question(question: str) -> str:
    q = question.strip().lower()
    rules = [
        ('how does it work', 'how does the retrieval pipeline work in LangChain'),
        ('prompt flow', 'prompt flow in retrieval augmented generation'),
        ('context', 'retrieved context and context compression'),
        ('rag', 'retrieval augmented generation'),
    ]
    rewritten = q
    for old, new in rules:
        if old in rewritten:
            rewritten = rewritten.replace(old, new)
    return rewritten

sample_question = 'How does it work for rag?'
print('Original :', sample_question)
print('Rewritten:', rewrite_question(sample_question))


## 4) Use the rewritten question for retrieval

A rewrite is not the final answer. It is the search query you use to fetch better context.

That is the main idea behind query rewriting prompt flows in RAG.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
chunks = splitter.split_documents(docs)

embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
store = FAISS.from_documents(chunks, embeddings)

query = rewrite_question(sample_question)
retrieved = store.similarity_search(query, k=3)

for i, doc in enumerate(retrieved, 1):
    print(f'--- Retrieved {i} ---')
    print(doc.page_content)
    print(doc.metadata)


## 5) Contextual compression

After retrieval, the raw chunks may be too long or noisy.

Context compression means keeping the useful parts and trimming the rest. The Deep Agents context-engineering docs describe this as part of managing growing context across long runs.


In [ ]:
def compress_context(retrieved_docs, max_chars_per_doc: int = 120):
    compressed = []
    for doc in retrieved_docs:
        text = doc.page_content.strip()
        if len(text) > max_chars_per_doc:
            text = text[:max_chars_per_doc].rsplit(' ', 1)[0] + '...'
        compressed.append(Document(page_content=text, metadata=doc.metadata))
    return compressed

compressed_docs = compress_context(retrieved, max_chars_per_doc=80)

for i, doc in enumerate(compressed_docs, 1):
    print(f'--- Compressed {i} ---')
    print(doc.page_content)
    print(doc.metadata)


## 6) Build a context block

A compact context block is easier for the model to use than a long raw dump of retrieved text.


In [ ]:
def build_context_block(docs):
    lines = []
    for i, doc in enumerate(docs, 1):
        src = doc.metadata.get('source', 'unknown')
        lines.append(f'[{i}] source={src}: {doc.page_content}')
    return '\n'.join(lines)

context_block = build_context_block(compressed_docs)
print(context_block)


## 7) Structured prompt templates

LangSmith’s prompt docs distinguish between prompts and prompt templates: prompts are what the model sees, while templates are reusable prompts with dynamic placeholders filled at runtime. The template-format guide says LangSmith supports reusable prompt templates with placeholders.


In [ ]:
retrieval_prompt_template = '''
You are a helpful assistant answering only from the provided context.

Question:
{question}

Context:
{context}

Instructions:
- Use only the context.
- If the context is incomplete, say so.
- Keep the answer short and direct.
'''

filled_prompt = retrieval_prompt_template.format(
    question=sample_question,
    context=context_block,
)

print(filled_prompt)


## 8) Why templates help in RAG

Templates make the retrieval prompt repeatable.

They are useful when you want to:
- swap questions at runtime
- test prompt versions
- keep the answer format consistent
- add variables like persona, style, or output length


In [ ]:
def make_retrieval_prompt(question: str, context: str, style: str = 'concise') -> str:
    template = '''
You are a {style} assistant.

Question:
{question}

Context:
{context}

Answer using only the context.
'''
    return template.format(style=style, question=question, context=context)

print(make_retrieval_prompt(sample_question, context_block, style='concise'))


## 9) Basic prompt flow for RAG

A simple prompt flow often looks like this:

1. Rewrite the question.
2. Retrieve relevant chunks.
3. Compress the retrieved context.
4. Format the final prompt with a template.
5. Send it to the model.

This is the basic version of the same idea used in hybrid RAG pipelines. The retrieval docs describe hybrid RAG as including query enhancement, retrieval validation, and answer validation.


In [ ]:
def rag_prompt_flow(question: str, vector_store):
    rewritten = rewrite_question(question)
    retrieved_docs = vector_store.similarity_search(rewritten, k=3)
    compressed = compress_context(retrieved_docs, max_chars_per_doc=80)
    context = build_context_block(compressed)
    prompt = make_retrieval_prompt(rewritten, context)
    return {
        'original_question': question,
        'rewritten_question': rewritten,
        'retrieved_count': len(retrieved_docs),
        'compressed_count': len(compressed),
        'prompt': prompt,
    }

flow_result = rag_prompt_flow('How does it work for rag?', store)
flow_result


## 10) Add a simple validation step

In a more advanced flow, you can validate whether the retrieved context is good enough before answering.

A very basic version can simply check whether the context contains a required keyword or topic.


In [ ]:
def validate_context(context: str, required_terms=None):
    required_terms = required_terms or []
    lower = context.lower()
    return all(term.lower() in lower for term in required_terms)

print(validate_context(context_block, required_terms=['retrieval', 'context']))
print(validate_context(context_block, required_terms=['billing']))


## 11) Compare the three pieces

- Query rewriting improves what you search for.
- Context compression improves what you send to the model.
- Prompt templates improve how you format the final request.

Together, they form a clean prompt-flow layer on top of RAG.


## Key takeaways

- Rewrite vague questions before retrieval.
- Compress retrieved text so the prompt stays small and relevant.
- Use prompt templates to keep retrieval prompts reusable.
- These are lightweight building blocks for hybrid RAG and more advanced agentic workflows.


## References

- Rewrite question: https://docs.langchain.com/oss/python/langgraph/agentic-rag#5-rewrite-question
- Hybrid RAG: https://docs.langchain.com/oss/python/langchain/retrieval#hybrid-rag
- Context compression: https://docs.langchain.com/oss/python/deepagents/context-engineering#context-compression
- Prompts vs. prompt templates: https://docs.langchain.com/langsmith/prompt-engineering-concepts#prompts-vs-prompt-templates
- Prompt template format: https://docs.langchain.com/langsmith/prompt-template-format
